In [3]:
import torch
import torch.nn as nn


def build_mlps(c_in, mlp_channels=None, ret_before_act=False, without_norm=False):
    layers = []
    num_layers = len(mlp_channels)

    for k in range(num_layers):
        if k + 1 == num_layers and ret_before_act:
            layers.append(nn.Linear(c_in, mlp_channels[k], bias=True))
        else:
            if without_norm:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=True), nn.ReLU()]) 
            else:
                layers.extend([nn.Linear(c_in, mlp_channels[k], bias=False), nn.BatchNorm1d(mlp_channels[k]), nn.ReLU()])
            c_in = mlp_channels[k]

    return nn.Sequential(*layers)



In [1]:
# Motion Transformer (MTR): https://arxiv.org/abs/2209.13508
# Published at NeurIPS 2022
# Written by Shaoshuai Shi 
# All Rights Reserved


import torch
import torch.nn as nn
import common_layers


class PointNetPolylineEncoder(nn.Module):
    def __init__(self, in_channels, hidden_dim, num_layers=3, num_pre_layers=1, out_channels=None):
        super().__init__()
        self.pre_mlps = common_layers.build_mlps(
            c_in=in_channels,
            mlp_channels=[hidden_dim] * num_pre_layers,
            ret_before_act=False
        )
        self.mlps = common_layers.build_mlps(
            c_in=hidden_dim * 2,
            mlp_channels=[hidden_dim] * (num_layers - num_pre_layers),
            ret_before_act=False
        )
        
        if out_channels is not None:
            self.out_mlps = common_layers.build_mlps(
                c_in=hidden_dim, 
                mlp_channels=[hidden_dim, out_channels], 
                ret_before_act=True, 
                without_norm=True
            )
        else:
            self.out_mlps = None 

    def forward(self, polylines, polylines_mask):
        """
        Args:
            polylines (batch_size, num_polylines, num_points_each_polylines, C):
            polylines_mask (batch_size, num_polylines, num_points_each_polylines):

        Returns:
        """
        batch_size, num_polylines,  num_points_each_polylines, C = polylines.shape
        print(polylines.shape, polylines_mask, polylines[polylines_mask].shape)
        # pre-mlp
        polylines_feature_valid = self.pre_mlps(polylines[polylines_mask])  
        print("polylines_feature_valid: ", polylines_feature_valid.shape, polylines_feature_valid)
        # (N, C) 
        # polylines[polylines_mask]：polylines 是输入的折线数据，
        # 形状为 (batch_size, num_polylines, num_points_each_polylines, C)；
        # polylines_mask 是对应的掩码，形状为 (batch_size, num_polylines, num_points_each_polylines)，
        # 它是一个布尔类型的张量，用于标记哪些点是有效的。
        polylines_feature = polylines.new_zeros(batch_size, num_polylines,  num_points_each_polylines, polylines_feature_valid.shape[-1])
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        polylines_feature[polylines_mask] = polylines_feature_valid
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        # get global feature
        a = polylines_feature.max(dim=2)
        print("a: ", a)
        pooled_feature = polylines_feature.max(dim=2)[0]
        print("pooled_feature: ", pooled_feature.shape, pooled_feature)
        polylines_feature = torch.cat((polylines_feature, pooled_feature[:, :, None, :].repeat(1, 1, num_points_each_polylines, 1)), dim=-1)
        print("polylines_feature: ", polylines_feature.shape, polylines_feature)
        # mlp
        polylines_feature_valid = self.mlps(polylines_feature[polylines_mask])
        print("polylines_feature_valid: ", polylines_feature_valid.shape, polylines_feature_valid)
        feature_buffers = polylines_feature.new_zeros(batch_size, num_polylines, num_points_each_polylines, polylines_feature_valid.shape[-1])
        feature_buffers[polylines_mask] = polylines_feature_valid
        print("feature_buffers: ", feature_buffers.shape, feature_buffers)

        # max-pooling
        feature_buffers = feature_buffers.max(dim=2)[0]  # (batch_size, num_polylines, C)
        print("feature_buffers: ", feature_buffers.shape, feature_buffers)
        # out-mlp 
        if self.out_mlps is not None:
            valid_mask = (polylines_mask.sum(dim=-1) > 0)
            feature_buffers_valid = self.out_mlps(feature_buffers[valid_mask])  # (N, C)
            feature_buffers = feature_buffers.new_zeros(batch_size, num_polylines, feature_buffers_valid.shape[-1])
            feature_buffers[valid_mask] = feature_buffers_valid
        return feature_buffers


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 定义测试数据集类
class TestDataset(Dataset):
    def __init__(self, num_samples, num_polylines, num_points_each_polylines, in_channels):
        self.num_samples = num_samples
        self.num_polylines = num_polylines
        self.num_points_each_polylines = num_points_each_polylines
        self.in_channels = in_channels

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        polylines = torch.randn(self.num_polylines, self.num_points_each_polylines, self.in_channels)
        polylines_mask = torch.randint(0, 2, (self.num_polylines, self.num_points_each_polylines)).bool()
        return polylines, polylines_mask


# 定义测试函数
def test_PointNetPolylineEncoder():
    in_channels = 5
    hidden_dim = 10
    num_layers = 3
    num_pre_layers = 1
    out_channels = 8
    batch_size = 1

    # 创建 PointNetPolylineEncoder 实例
    encoder = PointNetPolylineEncoder(
        in_channels=in_channels,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_pre_layers=num_pre_layers,
        out_channels=out_channels
    )

    # 创建测试数据集和数据加载器
    dataset = TestDataset(
        num_samples=batch_size,
        num_polylines=3,
        num_points_each_polylines=4,
        in_channels=in_channels
    )
    dataloader = DataLoader(dataset, batch_size=batch_size)

    # 进行测试
    for polylines, polylines_mask in dataloader:
        print("Input shape:", polylines.shape, polylines_mask.shape)
        output = encoder(polylines, polylines_mask)
        print("Output shape:", output.shape)
        assert len(output.shape) == 3
        assert output.shape[0] == batch_size
        break


if __name__ == "__main__":
    test_PointNetPolylineEncoder()

Input shape: torch.Size([1, 3, 4, 5]) torch.Size([1, 3, 4])
torch.Size([1, 3, 4, 5]) tensor([[[False,  True, False,  True],
         [False, False,  True,  True],
         [ True, False, False,  True]]]) torch.Size([6, 5])
polylines_feature_valid:  torch.Size([6, 10]) tensor([[0.0000, 0.0000, 0.0000, 1.7813, 1.0903, 1.3545, 0.0000, 1.7026, 0.0000,
         1.3493],
        [0.5508, 0.6120, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.3299,
         0.0000],
        [0.0000, 0.0000, 1.2012, 0.8096, 0.0000, 0.0000, 1.5064, 0.1200, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.4502, 1.1726, 0.0000, 0.6524, 1.1995,
         1.4455],
        [1.4608, 1.3281, 0.0000, 0.0000, 0.2464, 0.2639, 0.4241, 0.0000, 0.0000,
         0.0000],
        [0.2181, 0.3193, 1.4342, 0.0000, 0.0000, 0.0000, 0.9038, 0.0000, 0.0000,
         0.0000]], grad_fn=<ReluBackward0>)
polylines_feature:  torch.Size([1, 3, 4, 10]) tensor([[[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0.